# SHAP Explainability

This notebook explains SHAP (Shapley Additive exPlanations) in a simple way with practical code.

You will learn:
- What SHAP values are
- Why they matter in ML projects
- How to interpret SHAP values (global and local)
- Types of SHAP explainers: **Kernel**, **Tree**, **Linear**, and **Deep**
- Where each type is used in real projects

## 1) Why Explainability Matters in ML

In real ML projects, accuracy is not enough.

Explainability helps you:
- build trust with users/stakeholders
- debug wrong model behavior
- detect data leakage and bias
- support compliance/audits (finance, healthcare, insurance)
- choose better features for next model iteration

SHAP is one of the most popular methods because it gives **consistent feature contribution values** for each prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression, make_classification
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import shap

np.random.seed(42)
shap.initjs()

## 2) SHAP Intuition (Simple)

SHAP comes from game theory (Shapley values).

Think of a prediction as a team result:
- baseline prediction = average model output
- each feature contributes some amount
- all contributions add up to final prediction

Mathematically (for one sample):

`prediction = base_value + sum(SHAP feature contributions)`

Interpretation:
- positive SHAP value: pushes prediction higher
- negative SHAP value: pushes prediction lower
- larger absolute value: stronger influence

## 3) Example: Tree Model + SHAP Values (Regression)

We will train a Random Forest regressor and explain its predictions using Tree SHAP.

In [ ]:
# Create sample regression data
X, y = make_regression(
    n_samples=1200,
    n_features=10,
    n_informative=7,
    noise=15,
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(X.shape[1])]
X = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
rf_reg = RandomForestRegressor(n_estimators=300, random_state=42)
rf_reg.fit(X_train, y_train)

# Explain with TreeExplainer
tree_explainer_reg = shap.TreeExplainer(rf_reg)
shap_values_reg = tree_explainer_reg(X_test)

print("SHAP values shape:", shap_values_reg.values.shape)
print("Base value shape:", np.array(shap_values_reg.base_values).shape)
print("First prediction:", rf_reg.predict(X_test.iloc[[0]])[0])

In [ ]:
# Global explanation: which features matter most overall?
shap.plots.beeswarm(shap_values_reg, max_display=10)

In [ ]:
# Local explanation: why one specific prediction is high/low?
sample_idx = 0
shap.plots.waterfall(shap_values_reg[sample_idx], max_display=10)

base = shap_values_reg.base_values[sample_idx]
phi_sum = shap_values_reg.values[sample_idx].sum()
pred = rf_reg.predict(X_test.iloc[[sample_idx]])[0]

print(f"Base value: {base:.4f}")
print(f"Sum of SHAP contributions: {phi_sum:.4f}")
print(f"Base + contributions: {(base + phi_sum):.4f}")
print(f"Model prediction: {pred:.4f}")

## 4) Types of SHAP Explainers

### A) Tree SHAP
Use for tree-based models:
- Random Forest
- XGBoost
- LightGBM
- CatBoost

Why use it:
- fast and accurate for tree models
- most common SHAP type in industry

### B) Linear SHAP
Use for linear models:
- Linear Regression
- Logistic Regression

Why use it:
- very fast
- coefficients and SHAP are easy to relate

### C) Kernel SHAP
Model-agnostic (works with almost any model):
- SVM
- custom black-box model
- any model with a predict function

Why use it:
- very flexible

Trade-off:
- slower than Tree/Linear SHAP

### D) Deep SHAP
Use for deep learning models:
- TensorFlow / Keras
- PyTorch (via specific support)

Why use it:
- explains neural network predictions
- good for tabular/deep tasks when deep model is needed

In [ ]:
# -------- Linear SHAP Example --------
X_lin, y_lin = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42
)

lin_features = [f"lin_f{i}" for i in range(X_lin.shape[1])]
X_lin = pd.DataFrame(X_lin, columns=lin_features)

X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin, y_lin, test_size=0.2, random_state=42
)

lin_model = LogisticRegression(max_iter=1000)
lin_model.fit(X_train_lin, y_train_lin)

lin_explainer = shap.LinearExplainer(lin_model, X_train_lin)
lin_shap_values = lin_explainer(X_test_lin)

shap.plots.bar(lin_shap_values, max_display=8)
print("Linear SHAP done.")

In [ ]:
# -------- Kernel SHAP Example (Model-Agnostic) --------
X_ker, y_ker = make_classification(
    n_samples=700,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42
)

ker_features = [f"ker_f{i}" for i in range(X_ker.shape[1])]
X_ker = pd.DataFrame(X_ker, columns=ker_features)

X_train_ker, X_test_ker, y_train_ker, y_test_ker = train_test_split(
    X_ker, y_ker, test_size=0.2, random_state=42
)

ker_model = LogisticRegression(max_iter=1000)
ker_model.fit(X_train_ker, y_train_ker)

# KernelExplainer expects a function that takes array-like input and returns predictions.
def predict_proba_class1(data):
    data_df = pd.DataFrame(data, columns=ker_features)
    return ker_model.predict_proba(data_df)[:, 1]

background = shap.sample(X_train_ker, 50, random_state=42)
kernel_explainer = shap.KernelExplainer(predict_proba_class1, background)

# Keep sample small because Kernel SHAP is expensive.
X_small = X_test_ker.iloc[:20]
kernel_shap_values = kernel_explainer.shap_values(X_small, nsamples=100)

# Legacy plotting API works well here for numpy/list output.
shap.summary_plot(kernel_shap_values, X_small, feature_names=ker_features)
print("Kernel SHAP done (for 20 samples).")

In [ ]:
# -------- Tree SHAP Example (Classification) --------
X_tree, y_tree = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42
)

tree_features = [f"tree_f{i}" for i in range(X_tree.shape[1])]
X_tree = pd.DataFrame(X_tree, columns=tree_features)

X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree, y_tree, test_size=0.2, random_state=42
)

tree_model = RandomForestClassifier(n_estimators=250, random_state=42)
tree_model.fit(X_train_tree, y_train_tree)

tree_explainer_cls = shap.TreeExplainer(tree_model)
tree_shap_values = tree_explainer_cls.shap_values(X_test_tree)

# Handle different SHAP output formats across versions.
if isinstance(tree_shap_values, list):
    tree_values_class1 = tree_shap_values[1]
elif isinstance(tree_shap_values, np.ndarray) and tree_shap_values.ndim == 3:
    tree_values_class1 = tree_shap_values[:, :, 1]
else:
    tree_values_class1 = tree_shap_values

shap.summary_plot(tree_values_class1, X_test_tree, feature_names=tree_features, plot_type="bar")
print("Tree SHAP (classification) done.")

In [ ]:
# -------- Deep SHAP Example (Neural Network) --------
# This block needs tensorflow. If tensorflow is not installed, it will print a helpful message.

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Dense

    X_deep, y_deep = make_classification(
        n_samples=1200,
        n_features=12,
        n_informative=8,
        n_redundant=1,
        random_state=42
    )

    deep_features = [f"deep_f{i}" for i in range(X_deep.shape[1])]
    X_deep = pd.DataFrame(X_deep, columns=deep_features)

    X_train_deep, X_test_deep, y_train_deep, y_test_deep = train_test_split(
        X_deep, y_deep, test_size=0.2, random_state=42
    )

    # Simple neural network for binary classification
    deep_model = Sequential([
        Dense(32, activation="relu", input_shape=(X_train_deep.shape[1],)),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    deep_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    deep_model.fit(X_train_deep, y_train_deep, epochs=8, batch_size=32, verbose=0)

    background_deep = X_train_deep.iloc[:100].astype("float32").values
    explain_deep = X_test_deep.iloc[:20].astype("float32").values

    deep_explainer = shap.DeepExplainer(deep_model, background_deep)
    deep_shap_values = deep_explainer.shap_values(explain_deep)

    # For binary output, SHAP may return list of one array.
    if isinstance(deep_shap_values, list):
        deep_values_plot = deep_shap_values[0]
    else:
        deep_values_plot = deep_shap_values

    shap.summary_plot(deep_values_plot, explain_deep, feature_names=deep_features)
    print("Deep SHAP done.")

except ImportError:
    print("TensorFlow is not installed. Run: pip install tensorflow")
except Exception as e:
    print("Deep SHAP example could not run in this environment:")
    print(e)

## 5) How to Choose the Right SHAP Explainer

Use this quick decision guide:

- If your model is tree-based (XGBoost, LightGBM, CatBoost, RandomForest): use **TreeExplainer**
- If your model is linear (Linear/Logistic Regression): use **LinearExplainer**
- If your model is deep learning (Keras/TensorFlow/PyTorch): use **DeepExplainer**
- If none of the above / custom black-box: use **KernelExplainer**

Rule of thumb:
- prefer specialized explainers first (Tree/Linear/Deep)
- use Kernel SHAP when you need model-agnostic explanation

## 6) Why SHAP Values Are Important in ML Projects

1. Model debugging:
- Detect when model uses wrong or noisy features
- Catch leakage (feature that should not be available at prediction time)

2. Business trust:
- Explain "why this customer was rejected/approved"
- Improve communication between data team and business team

3. Fairness & risk checks:
- Inspect influence of sensitive attributes (direct or proxy)
- Support responsible AI and compliance reviews

4. Feature engineering feedback:
- Find top useful features globally
- Remove low-impact features to simplify model

5. Monitoring over time:
- Compare SHAP distributions in production to detect drift

## 7) Common Mistakes and Best Practices

Common mistakes:
- Interpreting SHAP as causal effect (it is contribution, not causality)
- Using very small/unrepresentative background data
- Comparing SHAP values across models with different outputs without care
- Running Kernel SHAP on huge datasets (too slow)

Best practices:
- Start with a small representative sample
- Use Tree/Linear explainer when possible (faster and stable)
- Analyze both global (beeswarm/bar) and local (waterfall/force) plots
- Validate explanations with domain knowledge
- Log SHAP summaries in model evaluation reports

In [ ]:
# Optional: quick helper to estimate global importance from SHAP values

def mean_abs_shap_importance(shap_values, feature_names):
    """Return mean absolute SHAP importance as a sorted DataFrame."""
    values = np.abs(shap_values.values).mean(axis=0)
    out = pd.DataFrame({"feature": feature_names, "mean_abs_shap": values})
    return out.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

importance_df = mean_abs_shap_importance(shap_values_reg, feature_names)
importance_df.head(10)

## 8) Final Takeaway

SHAP gives a clear answer to:
- **Globally:** which features matter most across the dataset?
- **Locally:** why did this one specific prediction happen?

If you remember one thing:
- SHAP values decompose prediction into feature contributions.
- This makes models easier to trust, debug, and improve in real projects.